In [ ]:
# @title
# -*- coding: utf-8 -*-
"""
cbec build package v47  for Github

CBEC JTAER package builder.
Frozen Route A architecture, notation-harmonized package.

Exports:
- CBEC_EU27_v47_backbone_2023.csv
- CBEC_EU27_v47_overlays_2023.csv
- CBEC_EU27_v47_validation_2023.csv
- CBEC_EU27_v47_robustness_2023.csv
- CBEC_EU27_v47_extraction_audit_2023.csv
- CBEC_EU27_v47_master_2023.xlsx
- CBEC_EU27_v47_metadata_2023.xlsx
- CBEC_EU27_v47_All_Files.zip
"""

import subprocess
import sys
import os
import base64
import zipfile
from typing import Dict, List, Tuple, Optional


def install_deps() -> None:
    pkgs = ["pandas", "numpy", "eurostat", "ipywidgets", "openpyxl"]
    for p in pkgs:
        try:
            __import__(p)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])


install_deps()

import pandas as pd
import numpy as np
import eurostat
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

clear_output()

PACKAGE_VERSION = "v47_final"
TARGET_YEAR = 2023
PACKAGE_PREFIX = "CBEC_EU27_v47"

EU27 = [
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL", "ES",
    "FI", "FR", "HR", "HU", "IE", "IT", "LT", "LU", "LV", "MT",
    "NL", "PL", "PT", "RO", "SE", "SI", "SK"
]

# Official 2023 World Bank LPI values as transcribed into the project package.
OFFICIAL_LPI_2023 = {
    "AT": 4.0, "BE": 4.0, "BG": 3.2, "CY": 3.2, "CZ": 3.3, "DE": 4.1, "DK": 4.1,
    "EE": 3.4, "EL": 3.4, "ES": 3.9, "FI": 4.0, "FR": 3.9, "HR": 3.3, "HU": 3.3,
    "IE": 3.3, "IT": 3.7, "LT": 3.4, "LU": 3.4, "LV": 3.4, "MT": 3.2, "NL": 4.0,
    "PL": 3.4, "PT": 3.3, "RO": 3.2, "SE": 4.0, "SI": 3.3, "SK": 3.3
}

AUDIT_LOG: List[Dict[str, object]] = []


def filter_exact_eu27(df: pd.DataFrame, country_col: str = "geo\\TIME_PERIOD") -> pd.DataFrame:
    if country_col not in df.columns:
        country_col = next((c for c in df.columns if "geo" in str(c).lower()), None)
    if country_col is None:
        raise KeyError("No geo-like column found in dataframe.")
    dff = df[df[country_col].isin(EU27)].copy()
    dff = dff.rename(columns={country_col: "region_code"})
    return dff.sort_values("region_code").reset_index(drop=True)


def ensure_eu27_base(df: Optional[pd.DataFrame], value_cols: List[str]) -> pd.DataFrame:
    base = pd.DataFrame({"region_code": EU27})
    if df is None or df.empty:
        for col in value_cols:
            base[col] = np.nan
        return base
    return base.merge(df, on="region_code", how="left")


def log_audit(variable: str, dataset: str, code: str, year: int, filters: object,
              usable_count: int, missing: List[str], status: str, notes: str) -> None:
    AUDIT_LOG.append({
        "variable": variable,
        "dataset": dataset,
        "code": code,
        "year_requested": year,
        "year_obtained": year,
        "filters": str(filters),
        "usable_count": usable_count,
        "usable_share": round(usable_count / len(EU27), 4),
        "missing_countries": ", ".join(missing) if missing else "None",
        "status": status,
        "notes": notes,
    })


def make_download_link(path: str, label: str, color: str) -> str:
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    mime = (
        "application/zip" if path.endswith(".zip")
        else "text/csv" if path.endswith(".csv")
        else "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    )
    return (
        f'<a download="{os.path.basename(path)}" href="data:{mime};base64,{b64}" '
        f'style="background-color:{color}; color:white; padding:8px 16px; margin-right:10px; '
        f'text-decoration:none; border-radius:4px; font-weight:bold; font-size:13px; '
        f'box-shadow:0 2px 4px rgba(0,0,0,0.1); display:inline-block; margin-bottom:5px;">{label}</a>'
    )


def status_from_count(count: int) -> str:
    if count == len(EU27):
        return "COMPLETE"
    if count == 0:
        return "FAILED"
    return "PARTIAL"


def extract_single_series(ds: str, filters: Dict[str, str], year: int, out_col: str,
                          variable: str, code: str, notes: str) -> pd.DataFrame:
    try:
        df = eurostat.get_data_df(ds)
        for k, v in filters.items():
            if k not in df.columns:
                raise KeyError(f"Column {k} not found in {ds}")
            df = df[df[k] == v]
        dff = filter_exact_eu27(df)
        if str(year) not in dff.columns:
            raise KeyError(f"Year {year} not found in {ds}")
        out = dff[["region_code", str(year)]].rename(columns={str(year): out_col})
        out[out_col] = pd.to_numeric(out[out_col], errors="coerce")
        out = ensure_eu27_base(out, [out_col])
        missing = out[out[out_col].isna()]["region_code"].tolist()
        log_audit(variable, ds, code, year, filters, len(EU27) - len(missing), missing,
                  status_from_count(len(EU27) - len(missing)), notes)
        return out
    except Exception as e:
        log_audit(variable, ds, code, year, filters, 0, EU27.copy(), "FAILED", str(e))
        return ensure_eu27_base(None, [out_col])


def fetch_esp(year: int) -> pd.DataFrame:
    return extract_single_series(
        ds="isoc_ec_esels",
        filters={"freq": "A", "size_emp": "GE10", "nace_r2": "C10-S951_X_K", "indic_is": "E_AESELL", "unit": "PC_ENT"},
        year=year,
        out_col="esp_pct",
        variable="ESP",
        code="E_AESELL",
        notes="Enterprise e-sales penetration"
    )


def fetch_dfp(year: int) -> pd.DataFrame:
    return extract_single_series(
        ds="tin00099",
        filters={"freq": "A", "indic_is": "I_IUBK", "unit": "PC_IND", "ind_type": "IND_TOTAL"},
        year=year,
        out_col="dfp_pct",
        variable="DFP",
        code="I_IUBK",
        notes="Digital financial participation proxy"
    )


def fetch_lpi(year: int) -> pd.DataFrame:
    df = pd.DataFrame({"region_code": list(OFFICIAL_LPI_2023.keys()), "lpi_score": list(OFFICIAL_LPI_2023.values())})
    df = ensure_eu27_base(df, ["lpi_score"])
    log_audit("LPI", "World_Bank_LPI_2023", "LPI.OVRL.XQ", year, {}, len(EU27), [], "COMPLETE",
              "Official 2023 LPI values embedded in package for stable reproducibility")
    return df


def fetch_cbo(year: int) -> pd.DataFrame:
    return extract_single_series(
        ds="isoc_ec_ibos",
        filters={"freq": "A", "ind_type": "IND_TOTAL", "indic_is": "I_BPG_EU", "unit": "PC_IND_BUY3"},
        year=year,
        out_col="cbo_pct",
        variable="CBO",
        code="I_BPG_EU",
        notes="Cross-border buying openness overlay"
    )


def _extract_mp_population(year: int) -> Tuple[pd.DataFrame, str]:
    ds = "demo_pjan"
    candidates = [
        {"freq": "A", "sex": "T", "age": "Y16-74", "unit": "NR"},
        {"freq": "A", "sex": "T", "age": "TOTAL", "unit": "NR"},
    ]
    df = eurostat.get_data_df(ds)
    for filters in candidates:
        dff = df.copy()
        valid = True
        for k, v in filters.items():
            if k not in dff.columns:
                valid = False
                break
            dff = dff[dff[k] == v]
        if not valid:
            continue
        dff = filter_exact_eu27(dff)
        if str(year) not in dff.columns:
            continue
        out = dff[["region_code", str(year)]].rename(columns={str(year): "mp_population_base"})
        out["mp_population_base"] = pd.to_numeric(out["mp_population_base"], errors="coerce")
        out = ensure_eu27_base(out, ["mp_population_base"])
        if out["mp_population_base"].notna().sum() == len(EU27):
            return out, filters["age"]
    return ensure_eu27_base(None, ["mp_population_base"]), "FAILED"


def fetch_mp(year: int) -> pd.DataFrame:
    try:
        pop_df, age_used = _extract_mp_population(year)
        inc_df = extract_single_series(
            ds="tin00096",
            filters={"freq": "A", "indic_is": "I_BLT12", "unit": "PC_IND", "ind_type": "IND_TOTAL"},
            year=year,
            out_col="mp_online_shopping_pct",
            variable="MP_INC",
            code="I_BLT12",
            notes="Online-shopping incidence used in MP overlay"
        )

        mp = ensure_eu27_base(None, []).merge(pop_df, on="region_code", how="left").merge(inc_df, on="region_code", how="left")
        mp["mp_estimated_eshoppers"] = (mp["mp_population_base"] * mp["mp_online_shopping_pct"]) / 100
        mp["mp_formula_label"] = f"population_{age_used} * online_shopping_incidence / 100"
        mp["mp_population_age_used"] = age_used

        missing = mp[mp["mp_estimated_eshoppers"].isna()]["region_code"].tolist()
        log_audit("MP", "demo_pjan + tin00096", f"population_age={age_used} + I_BLT12", year,
                  {"population_age_used": age_used, "incidence_code": "I_BLT12"},
                  len(EU27) - len(missing), missing, status_from_count(len(EU27) - len(missing)),
                  "Market potential overlay")
        return mp
    except Exception as e:
        log_audit("MP", "demo_pjan + tin00096", "MP", year, {}, 0, EU27.copy(), "FAILED", str(e))
        out = pd.DataFrame({
            "region_code": EU27,
            "mp_population_base": np.nan,
            "mp_online_shopping_pct": np.nan,
            "mp_estimated_eshoppers": np.nan,
            "mp_formula_label": "FAILED",
            "mp_population_age_used": "FAILED",
        })
        return out


def fetch_eti(year: int) -> pd.DataFrame:
    ds = "isoc_ec_evals"
    ind = "E_AWSVAL"
    base_filters = {"freq": "A", "size_emp": "GE10", "nace_r2": "C10-S951_X_K", "indic_is": ind}
    base = ensure_eu27_base(None, ["eti_pct"])
    base["eti_dataset_code"] = "FAILED"

    try:
        df = eurostat.get_data_df(ds)
        if df is None or df.empty:
            raise ValueError(f"Dataset {ds} is empty or unreachable.")

        def extract_unit(unit_name: str) -> Optional[pd.DataFrame]:
            filters = base_filters.copy()
            filters["unit"] = unit_name
            dff = df.copy()
            for k, v in filters.items():
                if k not in dff.columns:
                    return None
                dff = dff[dff[k] == v]
            if dff.empty:
                return None
            dff = filter_exact_eu27(dff)
            if str(year) not in dff.columns:
                return None
            out = dff[["region_code", str(year)]].rename(columns={str(year): "eti_pct"})
            out["eti_pct"] = pd.to_numeric(out["eti_pct"], errors="coerce")
            out = ensure_eu27_base(out, ["eti_pct"])
            out["eti_dataset_code"] = f"{ds} ({unit_name})"
            return out

        primary = extract_unit("PC_TURN")
        result = primary.copy() if primary is not None else base.copy()

        fallback = extract_unit("PC_ETURN")
        if fallback is not None:
            mask = result["eti_pct"].isna()
            fill_map = fallback.set_index("region_code")["eti_pct"]
            result.loc[mask, "eti_pct"] = result.loc[mask, "region_code"].map(fill_map)
            if mask.any():
                result.loc[mask, "eti_dataset_code"] = f"{ds} (PC_ETURN fallback)"
                result.loc[~mask, "eti_dataset_code"] = result.loc[~mask, "eti_dataset_code"].fillna(f"{ds} (PC_TURN)")

        missing = result[result["eti_pct"].isna()]["region_code"].tolist()
        log_audit("ETI", ds, ind, year, {"base": base_filters, "primary_unit": "PC_TURN", "fallback_unit": "PC_ETURN"},
                  len(EU27) - len(missing), missing, status_from_count(len(EU27) - len(missing)),
                  "Enterprise-side outcome-consistency route")
        return result
    except Exception as e:
        log_audit("ETI", ds, ind, year, base_filters, 0, EU27.copy(), "FAILED", str(e))
        return base


def fetch_gdp_pps(year: int) -> pd.DataFrame:
    return extract_single_series(
        ds="tec00114",
        filters={"freq": "A", "indic_ppp": "VI_PPS_EU27_2020_HAB", "ppp_cat18": "GDP"},
        year=year,
        out_col="gdp_pps_index",
        variable="GDP_PPS",
        code="VI_PPS_EU27_2020_HAB",
        notes="Robustness-only wealth-mimic guardrail"
    )


def run_builder(year: int):
    global AUDIT_LOG
    AUDIT_LOG = []

    base_df = pd.DataFrame({
        "region_code": EU27,
        "country": EU27,
        "source_year": year,
        "package_version": PACKAGE_VERSION,
    })

    esp = fetch_esp(year)
    dfp = fetch_dfp(year)
    lpi = fetch_lpi(year)
    cbo = fetch_cbo(year)
    mp = fetch_mp(year)
    eti = fetch_eti(year)
    gdp = fetch_gdp_pps(year)

    df_backbone = base_df.merge(esp, on="region_code").merge(dfp, on="region_code").merge(lpi, on="region_code")
    df_overlays = base_df.merge(cbo, on="region_code").merge(mp, on="region_code")
    df_validation = base_df.merge(eti, on="region_code")
    df_robustness = base_df.merge(gdp, on="region_code")
    df_audit = pd.DataFrame(AUDIT_LOG)

    return df_backbone, df_overlays, df_validation, df_robustness, df_audit


def export_outputs(df_backbone: pd.DataFrame, df_overlays: pd.DataFrame, df_validation: pd.DataFrame,
                   df_robustness: pd.DataFrame, df_audit: pd.DataFrame, year: int) -> List[str]:
    files = [
        f"{PACKAGE_PREFIX}_backbone_{year}.csv",
        f"{PACKAGE_PREFIX}_overlays_{year}.csv",
        f"{PACKAGE_PREFIX}_validation_{year}.csv",
        f"{PACKAGE_PREFIX}_robustness_{year}.csv",
        f"{PACKAGE_PREFIX}_extraction_audit_{year}.csv",
        f"{PACKAGE_PREFIX}_master_{year}.xlsx",
        f"{PACKAGE_PREFIX}_metadata_{year}.xlsx",
    ]

    df_backbone.to_csv(files[0], index=False)
    df_overlays.to_csv(files[1], index=False)
    df_validation.to_csv(files[2], index=False)
    df_robustness.to_csv(files[3], index=False)
    df_audit.to_csv(files[4], index=False)

    with pd.ExcelWriter(files[5], engine="openpyxl") as writer:
        df_backbone.to_excel(writer, sheet_name="backbone", index=False)
        df_overlays.to_excel(writer, sheet_name="overlays", index=False)
        df_validation.to_excel(writer, sheet_name="validation", index=False)
        df_robustness.to_excel(writer, sheet_name="robustness", index=False)
        merged = (
            df_backbone
            .merge(df_overlays.drop(columns=["country", "source_year", "package_version"]), on="region_code")
            .merge(df_validation.drop(columns=["country", "source_year", "package_version"]), on="region_code")
            .merge(df_robustness.drop(columns=["country", "source_year", "package_version"]), on="region_code")
        )
        merged.to_excel(writer, sheet_name="merged_all", index=False)
        df_audit.to_excel(writer, sheet_name="audit", index=False)

    variable_dictionary = pd.DataFrame([
        {"abbr": "ESP", "column": "esp_pct", "role": "Backbone", "definition": "Enterprise e-sales penetration"},
        {"abbr": "DFP", "column": "dfp_pct", "role": "Backbone", "definition": "Digital financial participation proxy"},
        {"abbr": "LPI", "column": "lpi_score", "role": "Backbone", "definition": "Logistics performance index"},
        {"abbr": "CBO", "column": "cbo_pct", "role": "Overlay", "definition": "Cross-border buying openness"},
        {"abbr": "MP", "column": "mp_estimated_eshoppers", "role": "Overlay", "definition": "Market potential"},
        {"abbr": "ETI", "column": "eti_pct", "role": "Enterprise-side consistency", "definition": "E-commerce turnover intensity"},
        {"abbr": "GDP_PPS", "column": "gdp_pps_index", "role": "Robustness", "definition": "GDP per capita in PPS"},
        {"abbr": "PC1_EF", "column": "derived_in_analysis", "role": "Latent factor", "definition": "First PCA component interpreted as execution feasibility"},
    ])

    legacy_mapping = pd.DataFrame([
        {"old_label": "C2", "new_label": "ESP"},
        {"old_label": "C4", "new_label": "DFP"},
        {"old_label": "C5", "new_label": "LPI"},
        {"old_label": "C1", "new_label": "CBO"},
        {"old_label": "MP1", "new_label": "MP"},
        {"old_label": "VAL1", "new_label": "ETI"},
        {"old_label": "C6", "new_label": "GDP_PPS"},
    ])

    role_definition = pd.DataFrame([
        {"block": "Backbone", "definition": "Primary execution-feasibility inputs"},
        {"block": "Overlays", "definition": "Scale and openness kept outside the backbone"},
        {"block": "Enterprise-side consistency", "definition": "Partial 2023 criterion-consistency route"},
        {"block": "Robustness", "definition": "Wealth-mimic guardrail only"},
    ])

    formula_notes = pd.DataFrame([
        {"item": "MP", "definition": "population_base * online_shopping_incidence / 100"},
        {"item": "LPI", "definition": "Embedded official 2023 values used for stable reproducibility"},
    ])

    version_log = pd.DataFrame([
        {"package_version": PACKAGE_VERSION, "package_prefix": PACKAGE_PREFIX, "architecture": "Frozen v4.7 Route A", "note": "Notation harmonized; architecture unchanged"}
    ])

    with pd.ExcelWriter(files[6], engine="openpyxl") as writer:
        variable_dictionary.to_excel(writer, sheet_name="variable_dictionary", index=False)
        legacy_mapping.to_excel(writer, sheet_name="legacy_mapping", index=False)
        df_audit[["variable", "dataset", "code", "filters", "usable_count", "status", "notes"]].to_excel(writer, sheet_name="audit_summary", index=False)
        role_definition.to_excel(writer, sheet_name="role_definition", index=False)
        formula_notes.to_excel(writer, sheet_name="formula_notes", index=False)
        version_log.to_excel(writer, sheet_name="version_log", index=False)

    zip_name = f"{PACKAGE_PREFIX}_All_Files.zip"
    with zipfile.ZipFile(zip_name, "w") as zf:
        for f in files:
            zf.write(f)
    files.append(zip_name)
    return files


def render_summary(df_backbone: pd.DataFrame, df_overlays: pd.DataFrame, df_validation: pd.DataFrame,
                   df_robustness: pd.DataFrame) -> str:
    backbone_ok = df_backbone[["esp_pct", "dfp_pct", "lpi_score"]].notna().all().all()
    overlays_ok = df_overlays[["cbo_pct", "mp_estimated_eshoppers"]].notna().all().all()
    robustness_ok = df_robustness[["gdp_pps_index"]].notna().all().all()

    eti_count = int(df_validation["eti_pct"].notna().sum())
    eti_status = status_from_count(eti_count)
    if eti_status == "COMPLETE":
        eti_text = f"[COMPLETE | {eti_count}/27 usable]"
        eti_color = "green"
    elif eti_status == "PARTIAL":
        eti_text = f"[PARTIAL | {eti_count}/27 usable]"
        eti_color = "orange"
    else:
        eti_text = f"[FAILED | {eti_count}/27 usable]"
        eti_color = "red"

    html = f"""
    <div style="background:#fff; border:1px solid #ddd; padding:15px; border-radius:4px; font-family:sans-serif; margin-bottom:15px;">
        <h3 style="margin-top:0;">Run Summary ({TARGET_YEAR})</h3>
        <ul style="list-style-type:none; padding-left:0;">
            <li><b>Backbone (ESP, DFP, LPI):</b> <span style="color:{'green' if backbone_ok else 'red'}; font-weight:bold;">[{'COMPLETE' if backbone_ok else 'INCOMPLETE'}]</span></li>
            <li><b>Overlays (CBO, MP):</b> <span style="color:{'green' if overlays_ok else 'red'}; font-weight:bold;">[{'COMPLETE' if overlays_ok else 'INCOMPLETE'}]</span></li>
            <li><b>Enterprise-side consistency (ETI):</b> <span style="color:{eti_color}; font-weight:bold;">{eti_text}</span></li>
            <li><b>Robustness (GDP_PPS):</b> <span style="color:{'green' if robustness_ok else 'red'}; font-weight:bold;">[{'COMPLETE' if robustness_ok else 'INCOMPLETE'}]</span></li>
        </ul>
    </div>
    """

    if backbone_ok and overlays_ok and robustness_ok and eti_status == "PARTIAL":
        html += (
            "<div style='background:#fff8e1; border:1px solid #f1c40f; padding:10px; border-radius:4px; "
            "color:#8a6d3b; font-weight:bold; margin-bottom:15px;'>"
            "ETI is partially usable rather than failed. The package is still valid for the frozen Route A architecture because the backbone and overlays remain EU-27 complete and ETI is used only as a constrained enterprise-side consistency sample."
            "</div>"
        )
    elif not (backbone_ok and overlays_ok and robustness_ok):
        html += (
            "<div style='background:#fde8e8; border:1px solid #e74c3c; padding:10px; border-radius:4px; "
            "color:#c0392b; font-weight:bold; margin-bottom:15px;'>"
            "One or more mandatory backbone, overlay, or robustness variables failed. The package should not be used until that problem is fixed."
            "</div>"
        )
    return html


ui_header = widgets.HTML(f"""
<div style="background-color:#f8f9fa; border-left: 6px solid #27ae60; padding:15px; border-radius:4px; font-family:sans-serif;">
    <h2 style="margin:0; color:#2c3e50;">CBEC Package Builder: {PACKAGE_VERSION}</h2>
    <p style="margin:5px 0 0 0; color:#555; font-size:14px;">
    <b>Frozen route:</b> 2023 EU-27 Route A with harmonized notation.<br>
    Backbone = ESP + DFP + LPI. Overlays = CBO + MP. Enterprise-side consistency = ETI. Robustness = GDP_PPS.
    </p>
</div>
""")

btn_run = widgets.Button(
    description=f"▶ Build harmonized v4.7 package ({TARGET_YEAR})",
    button_style="success",
    layout=widgets.Layout(width="340px", height="45px", margin="15px 0"),
)
out_main = widgets.Output()


def on_run(_):
    with out_main:
        clear_output()
        btn_run.description = "Building package..."
        btn_run.disabled = True
        try:
            df_backbone, df_overlays, df_validation, df_robustness, df_audit = run_builder(TARGET_YEAR)
            files = export_outputs(df_backbone, df_overlays, df_validation, df_robustness, df_audit, TARGET_YEAR)

            display(HTML(render_summary(df_backbone, df_overlays, df_validation, df_robustness)))
            dl_html = "<div style='display:flex; flex-wrap:wrap; gap:10px;'>"
            for f in files:
                color = "#8e44ad" if f.endswith(".zip") else "#27ae60" if f.endswith(".csv") else "#2980b9"
                dl_html += make_download_link(f, f"↓ {os.path.basename(f)}", color)
            dl_html += "</div>"
            display(HTML("<h3 style='font-family:sans-serif;'>Downloads</h3>"))
            display(HTML(dl_html))

            display(HTML("<hr><h3 style='font-family:sans-serif;'>Preview: Backbone</h3>"))
            display(df_backbone.head(4))
            display(HTML("<h3 style='font-family:sans-serif;'>Preview: Overlays</h3>"))
            display(df_overlays.head(4))
            display(HTML("<h3 style='font-family:sans-serif;'>Preview: Enterprise-side consistency</h3>"))
            display(df_validation.head(4))
            display(HTML("<h3 style='font-family:sans-serif;'>Preview: Robustness</h3>"))
            display(df_robustness.head(4))
            display(HTML("<h3 style='font-family:sans-serif;'>Preview: Extraction audit</h3>"))
            display(df_audit)
        except Exception as e:
            display(HTML(f"<div style='background:#e74c3c; color:white; padding:15px; border-radius:4px;'><b>Pipeline error:</b> {str(e)}</div>"))
        finally:
            btn_run.description = f"▶ Build harmonized v4.7 package ({TARGET_YEAR})"
            btn_run.disabled = False


btn_run.on_click(on_run)

display(ui_header)
display(btn_run)
display(out_main)
